> **How to use this notebook.** Step 1 of 5. Needs nothing but this repository and an internet connection (it installs OpenMPI and the pinned Python packages). No data is read here.
>
> **Nurnet files (included).** The site list (`nurnet_nuraghi.shp`, 7,167 nuraghi with tower counts) and the per-site building stone file are Nurnet's data, included in `data/nurnet/`; 02 copies them to the names the configuration expects (`DATA_DIR/sites/`, `DATA_DIR/stone/`) and checks their SHA-256. For the latest version and source information contact Nurnet at nurnet.presidente@gmail.com. Details: [HOW_TO_USE.md](HOW_TO_USE.md#nurnet-files).
>
> Order: 01_setup, 02_inputs, 03_scenarios, 04_analysis_report, 05_audit (Jupyter or Google Colab). Full guide: [notebooks/HOW_TO_USE.md](HOW_TO_USE.md).

**Getting the data** (full table: `02_inputs.ipynb`, section 0; step by step: [HOW_TO_USE.md](HOW_TO_USE.md)).
Nurnet's site list and per-site building stone are included in `data/nurnet/` (for the latest version and source
information contact Nurnet at nurnet.presidente@gmail.com). Every other raw input is public: ARPAS lithology,
Copernicus DEM GLO-30 (AWS tiles), AGRIS soils (WFS), RAS rivers; `notebooks/fetch_data.py` downloads them from the
official sources and rebuilds the model files, or link them by hand (HOW_TO_USE.md, 'Linking the data yourself').
Derived inputs: the prebuilt pack (`DERIVED_URL`, when public) or `python notebooks/fetch_data.py --rebuild-derived`.

# 01 Setup: software environment

Nuraghe-construction ABM (the authors), code = the frozen snapshot of the full audited run (`code/`,
see `CODE_PROVENANCE.md`). This notebook prepares a Python that runs the model:

1. installs OpenMPI (Debian/Colab as root: `apt-get`), then the pinned package versions from `requirements-colab.txt`;
2. checks every version against the pins;
3. starts a 2-rank MPI test (`mpirun --allow-run-as-root --oversubscribe -n 2`) and imports the model code.

**Runtime (estimate).** The first install takes several minutes (repast4py and mpi4py compile against OpenMPI);
with the packages already installed, a few seconds.

**Pins.** `requirements-colab.txt` is the environment of the 31 Oct 2026 reproducibility package (Python 3.13.5).
If a package that was already imported changes version, the cell asks you to restart the kernel and run the
notebook again.

## Software requirements (installed by section 1)
From `code/requirements.txt` (minimum versions) and `notebooks/requirements-colab.txt` (the exact, tested versions):

| what | version | install |
|---|---|---|
| Python | 3.12+ (byte-identical reproduction; 3.10-3.11 differ in the last digits, see README); tested 3.13.5 | (Colab: preinstalled) |
| OpenMPI or MPICH, with `mpicc` / `mpicxx` | system package | Ubuntu / Colab: `apt-get install -y openmpi-bin libopenmpi-dev` (done here when running as root); macOS: `brew install open-mpi`; Windows: WSL2 + the Ubuntu command; conda: `conda install -c conda-forge openmpi mpi4py` |
| repast4py | >= 1.1.6; tested 1.3.1 | Collier & Ozik 2022, doi:[10.1109/WSC57314.2022.10015389](https://doi.org/10.1109/WSC57314.2022.10015389); guide: https://repast.github.io/repast4py.site/ |
| mpi4py | >= 3.1; tested 4.1.2 | |
| numpy, numba, pandas | >= 1.24 / 0.58 / 2.0; tested 2.5.4 / 0.68.0 / 3.0.6 | |
| geopandas, pyogrio, rasterio | >= 0.14 / 0.7 / 1.3; tested 1.2.0 / 0.13.0 / 1.5.2 | the wheels include GDAL (tested: GDAL 3.12.2); no system GDAL needed |
| pyyaml | >= 6.0; tested 6.0.3 | |
| also pinned | scipy 1.18.1, shapely 2.2.0, pyproj 3.8.0, matplotlib 3.11.2, networkx 3.7, reportlab 5.1.0, ... | 33 pins in `requirements-colab.txt` |

One command (what section 1 runs): `CC=mpicc CXX=mpicxx python -m pip install -r notebooks/requirements-colab.txt`.
Resources (measured): a 100-seed run of the headline scenario on 6 ranks peaks at about 6 GB (5.9 GB proportional,
7.6 GB resident) and takes about 6 minutes on the 8 GB machine of the runs; plan 1-3 GB per MPI rank for alliance
runs; about 2 GB of disk for the inputs. Details: [HOW_TO_USE.md, Software requirements](HOW_TO_USE.md#software-requirements).

**Expected results of this notebook:** section 2 `pinned packages: 33; mismatches: 0`; section 3 `rank 0 of 2` and
`rank 1 of 2`; section 4 `model code imports OK` and `Setup complete`.

## Config (the only cell you edit)

All paths are set at the top of this cell. With `"auto"` the notebook uses the repository layout: `code/`,
`config/`, `data/` (raw inputs, see `data/README.md`), `data/derived/cache/` (derived inputs, see
`data/derived/README.md`) and `outputs/` (everything the notebooks write). Environment variables `NB_REPO_DIR`,
`NB_DATA_DIR`, `NB_DERIVED_DIR`, `NB_CONFIG`, `NB_OUTPUTS_DIR`, `NB_WORK_DIR`, `NB_RESULTS_DIR`, `NB_LOCK` override
the defaults (for headless runs with `jupyter nbconvert --execute`).

**Google Colab:** the repository is cloned to `/content/Repast_Nuraghe` by the setup cell. Each notebook runs in its own Colab runtime, so put the data and outputs on Google Drive: `MOUNT_DRIVE = True`, `DATA_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data"`, `DERIVED_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data/derived/cache"`, `OUTPUTS_DIR = "/content/drive/MyDrive/Repast_Nuraghe/outputs"` (the same values in every notebook).

In [ ]:
import os

# Repository layout: the only lines you may need to change. "auto" = the default folder of the repository.
REPO_DIR = os.environ.get("NB_REPO_DIR", "auto")         # the cloned repository ("auto": the parent of notebooks/)
REPO_URL = "https://github.com/maltaweel/Repast_Nuraghe"  # Colab: cloned to /content/Repast_Nuraghe if not found
DATA_DIR = os.environ.get("NB_DATA_DIR", "auto")         # raw inputs = data_dir ("auto": REPO_DIR/data; see data/README.md)
DERIVED_DIR = os.environ.get("NB_DERIVED_DIR", "auto")   # cached derived inputs ("auto": REPO_DIR/data/derived/cache)
CONFIG = os.environ.get("NB_CONFIG", "auto")             # base config ("auto": REPO_DIR/config/config_full_audited.yaml)
OUTPUTS_DIR = os.environ.get("NB_OUTPUTS_DIR", "auto")   # everything written here ("auto": REPO_DIR/outputs)
WORK_DIR = os.environ.get("NB_WORK_DIR", "auto")         # fast local disk for code + cache copies ("auto": OUTPUTS_DIR/work)
RESULTS_DIR = os.environ.get("NB_RESULTS_DIR", "auto")   # where runs are kept ("auto": OUTPUTS_DIR/notebook_runs)
MOUNT_DRIVE = False                                      # Colab only: mount Google Drive at /content/drive
INSTALL = True                                           # install OpenMPI + pinned packages if this runtime lacks them
LOCK = os.environ.get("NB_LOCK") or None                 # optional flock file (shared servers only)

## 1. Find the repository and install
Locates the repository, sets the folders, installs what is missing, checks the versions.

In [ ]:
# Setup (no edits needed): find the repository, set the folders, install if needed, import the helpers in repro.py.
import os, sys, glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

def _is_repo(d):
    return os.path.isfile(os.path.join(d, "notebooks", "repro.py")) and os.path.isdir(os.path.join(d, "code"))

def _find_repo(p):
    cands = [p] if p != "auto" else [os.path.dirname(os.getcwd()), os.getcwd(), "/content/Repast_Nuraghe"]
    for c in cands:
        if _is_repo(c):
            return os.path.abspath(c)
    if IN_COLAB and p == "auto":                     # Colab opens only the notebook: clone the repository once
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Repast_Nuraghe"], check=True)
        return "/content/Repast_Nuraghe"
    raise FileNotFoundError("Repository not found (tried " + ", ".join(cands) + "). Set REPO_DIR in the config cell.")

REPO_DIR = _find_repo(REPO_DIR)
PACKAGE_DIR = REPO_DIR                                     # name used by the helper calls below
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))
import repro
if OUTPUTS_DIR == "auto":
    OUTPUTS_DIR = os.path.join(REPO_DIR, "outputs")
OUTPUTS_DIR = os.path.abspath(OUTPUTS_DIR)
L = repro.set_layout(repo=REPO_DIR, data=DATA_DIR, cache=DERIVED_DIR, config=CONFIG,
                     reference=os.path.join(OUTPUTS_DIR, "reference"))
DATA_DIR, DERIVED_DIR, CONFIG = L["data"], L["cache"], L["config"]
WORK_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "work") if WORK_DIR == "auto" else WORK_DIR)
RESULTS_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "notebook_runs") if RESULTS_DIR == "auto" else RESULTS_DIR)
for k in ("REPO_DIR", "DATA_DIR", "DERIVED_DIR", "CONFIG", "WORK_DIR", "RESULTS_DIR"):
    print(f"{k:12s}", globals()[k])
status = repro.ensure_env(install=INSTALL)
print("MPI:", repro.mpi_report(), "| Python", sys.version.split()[0], "| CPUs", os.cpu_count())
assert status == "ok", "Restart the kernel now (see message above), then run this notebook again."

## 2. Version check
Every pinned package must match exactly. `torch` is not pinned: repast4py imports it, but the model never uses it
(every random draw is numpy's PCG64 generator, seeded `default_rng([seed, site id, ...])`).

In [ ]:
bad = repro.check_versions()
assert not bad, "versions differ from requirements-colab.txt: rerun section 1 (INSTALL = True), then restart"
import numpy, numba, pandas, repast4py, mpi4py
print("numpy", numpy.__version__, "| numba", numba.__version__, "| pandas", pandas.__version__,
      "| repast4py", repast4py.__version__ if hasattr(repast4py, "__version__") else "?", "| mpi4py", mpi4py.__version__)

## 3. MPI smoke test
Two ranks must start and see each other. On Colab as root, `--allow-run-as-root` is required; `--oversubscribe` lets 2 ranks run on 2 vCPUs.

In [ ]:
import subprocess
r = subprocess.run(["mpirun", "--allow-run-as-root", "--oversubscribe", "-n", "2", sys.executable, "-c",
                    "from mpi4py import MPI; c = MPI.COMM_WORLD; print('rank', c.Get_rank(), 'of', c.Get_size())"],
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout)
assert r.returncode == 0 and "of 2" in r.stdout, "mpirun failed: see the output above"

## 4. The model code imports
The code in `code/` must be byte-identical to the repository version (SHA-256 in `notebooks/inputs_manifest.csv`,
group `code`).

In [ ]:
repro._check_code(PACKAGE_DIR)
r = subprocess.run([sys.executable, "-c", "import sys; sys.path.insert(0, 'code'); import nbcommon, model, alliance, lit, sea, phasing, pre_networks; print('model code imports OK')"],
                   cwd=PACKAGE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                   env=dict(os.environ, NUMBA_CACHE_DIR=os.path.join(WORK_DIR, "numba_cache"), PYTHONDONTWRITEBYTECODE="1"))
print(r.stdout[-3000:])
assert r.returncode == 0
print("Setup complete. Next: 02_inputs.ipynb (then 03, 04, 05_audit)")